# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided example for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library, referencing all data entities using their `@id` attributes as required by the Croissant schema.

### Dataset Source
The dataset source is a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's Croissant metadata and access its record sets and fields using their Croissant `@id`s.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Already as an object

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Explore available record sets and their fields (column descriptions).

We reference all entities by their `@id` as required by Croissant. You can view all record sets and their children fields below.

In [ ]:
# Enumerate all record sets and their field ids and labels
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"@id: {rs['@id']}  |  Name: {rs['name'] if 'name' in rs else ''}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        # Print field @id and label
        field_id = f if isinstance(f, str) else f.get('@id', str(f))
        label = f.get('name', '') if isinstance(f, dict) else ''
        print(f"    Field @id: {field_id}  | Label: {label}")
    print()

## 3. Data Extraction
Load the records from a selected record set (`@id`). Data is returned and referenced by record set and field `@id` as per the schema.

**Note:** You may need to check the output above and select a specific record set `@id` for further exploration. Here, we demonstrate loading all records from the *first* available record set for initial analysis.

In [ ]:
# List all available record set @ids
rs_ids = [rs['@id'] for rs in record_sets]
print("Available record set @ids:", rs_ids)

# Choose the first record set for demonstration
record_set_id = rs_ids[0]
print(f"Using record set: {record_set_id}\n")

# Load records into a DataFrame
records = list(dataset.records(record_set=record_set_id))  # Each record is a dict, keys are field @id
df = pd.DataFrame(records)
print("Columns (field @ids):\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing, such as filtering, normalizing, and grouping, using *field* `@id` as columns.

We first search for a numeric field. If none found, try a string field. For example, using a field like `cr:Coverage`, `cr:MolecularWeight`, or similar (choose the exact field from the previous cell's column list).

In [ ]:
# Print all numeric-like columns to help with selection
numeric_cols = df.select_dtypes(include=[float, int]).columns.tolist()
print("Numeric candidate field @ids:", numeric_cols)

# For this dataset, let's try one typical field by @id. You may need to check the column list if uncertain.
if numeric_cols:
    numeric_field = numeric_cols[0]  # Use the first numeric field for demonstration
else:
    raise Exception("No numeric field found in the record set for EDA.")
print(f"Selected numeric field @id for filtering: {numeric_field}")

threshold = 10  # Example threshold; adjust as appropriate for your field
# Filter records
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (showing first rows):")
print(filtered_df.head())

# Normalize the selected numeric field
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records (showing field and normalized value):")
print(filtered_df[[numeric_field, norm_col]].head())

# Try grouping by a non-numeric column, if available
string_cols = df.select_dtypes(include=[object]).columns.tolist()
group_field = None
for col in string_cols:
    if df[col].nunique() < len(df) // 2:  # Choose a low-cardinality string column
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}' (mean of {numeric_field}):")
    print(grouped_df.head())
else:
    print("No suitable non-numeric field for grouping in this record set.")

## 5. Visualization
Visualize the normalized numeric field and optionally group means (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the normalized numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[norm_col], kde=True)
plt.title(f"Distribution of Normalized '{numeric_field}'")
plt.xlabel(norm_col)
plt.ylabel("Frequency")
plt.show()

# If grouped_df exists, plot the group mean bar plot
if 'grouped_df' in locals() and group_field is not None:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, you have:
- Loaded the FAIR^2 dataset (Croissant) schema and records using their `@id` with `mlcroissant`.
- Explored the record sets and fields using their Croissant identifiers.
- Extracted tabular data from a record set for EDA and visualization.
- Demonstrated basic filtering, normalization, grouping, and visualization using field `@id` for proper, reproducible reference.

**Next Steps:**
- Modify field/record set `@id` variables as desired for deeper or alternative analyses.
- Use the Croissant schema and `mlcroissant` package to fully automate and share FAIR data analyses.